# Sri Lanka fire detections, 2018–2023

NASA FIRMS MODIS SP and VIIRS S-NPP SP. Counts represent satellite detections within the pilot bounding box (79.5°E–82°E, 5.8°N–10°N).

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
from fire import SOURCES, monthly

table, metrics = monthly('79.5,5.8,82,10')
pilot = table.loc['2018-01':'2023-12']
paired = pilot[list(SOURCES)].dropna()
print(f'{len(paired)} of 72 months have complete downloads for both products.')
display(pilot.head())


In [ ]:
import plotly.express as px

comparison = pilot[list(SOURCES)].rename(columns={
    'MODIS_SP': 'MODIS', 'VIIRS_SNPP_SP': 'VIIRS S-NPP'
})
px.line(comparison, title='Monthly hotspot detections in the Sri Lanka pilot',
        labels={'value': 'Detections', 'month': 'UTC month', 'variable': 'Product'}).show()


In [ ]:
display(metrics)
px.line(pilot[['VIIRS_SNPP_SP', 'harmonized']],
        title='VIIRS observations and VIIRS-scale activity',
        labels={'value': 'Detections', 'month': 'UTC month', 'variable': 'Series'}).show()


In [ ]:
from analytics import evaluate_overlap, historical_anomalies
from grid import daily_grid

holdout, validation = evaluate_overlap(pilot)
display(validation)
cells = daily_grid('79.5,5.8,82,10', '2018-01-01', '2023-12-31')
both = (cells[list(SOURCES)] > 0).all(axis=1)
print(f'{both.sum():,} of {len(cells):,} occupied grid-days have detections from both products.')
display(cells.head())
display(historical_anomalies(pilot.harmonized).tail(12))


A complete month has a successfully processed file for every UTC day. Successful requests can still contain satellite coverage gaps. VIIRS is the reference scale and has its own detection biases. The reported holdout error compares raw and calibrated MODIS against VIIRS over later paired months.